In [0]:
# Databricks notebook source
import mlflow.pyfunc

In [0]:
# 1. Load the Latest Registered Model from Unity Catalog

model_name = "gold.customer_churn_model"
model_version = "1" 
model_uri = f"models:/{model_name}/{model_version}"
loaded_model = mlflow.pyfunc.load_model(model_uri)

In [0]:
# 2. Read Feature Data for Prediction
df_features = spark.table("gold.ml_customer_features").toPandas()
feature_cols = ['frequency', 'monetary', 'avg_order_value']

In [0]:
# 3. Perform Batch Prediction
df_features['predicted_churn_risk'] = loaded_model.predict(df_features[feature_cols])

# Predict Probabilities (Risk Score)
if hasattr(loaded_model._model_impl, "predict_proba"):
    df_features['churn_probability'] = loaded_model._model_impl.predict_proba(df_features[feature_cols])[:, 1]

In [0]:
# 4. Save Prediction Output to Gold Layer for Dashboard & App Visualization
predictions_spark_df = spark.createDataFrame(df_features)
predictions_spark_df.write.format("delta").mode("overwrite").saveAsTable("gold.customer_churn_predictions")

print("Batch inference completed. Predictions stored in `gold.customer_churn_predictions`.")